Calcuate DEM per grid

In [5]:
from pathlib import Path
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats


In [6]:
# 500m 网格
base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000")
grid_path = base_dir / "Jiangsu_grid_500m_ID.gpkg"
grid = gpd.read_file(grid_path)

print("Grid CRS:", grid.crs)

# DEM 原始文件
dem_path = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/DEM_JIangsu/江苏省_DEM_30m分辨率_NASA数据.tif")
dem_src = rasterio.open(dem_path)
print("DEM CRS:", dem_src.crs)


Grid CRS: EPSG:4547
DEM CRS: EPSG:4326


In [7]:
# 检查是否要重投影 DEM
if dem_src.crs != grid.crs:
    print("Reprojecting DEM to match grid CRS:", grid.crs)

    # 输出路径
    dem_reproj_path = dem_path.parent / "江苏省_DEM_30m_reprojected.tif"

    transform, width, height = calculate_default_transform(
        dem_src.crs,
        grid.crs,
        dem_src.width,
        dem_src.height,
        *dem_src.bounds
    )

    kwargs = dem_src.meta.copy()
    kwargs.update({
        "crs": grid.crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    # 写入重投影后的 DEM
    with rasterio.open(dem_reproj_path, "w", **kwargs) as dst:
        for i in range(1, dem_src.count + 1):
            reproject(
                source=rasterio.band(dem_src, i),
                destination=rasterio.band(dst, i),
                src_transform=dem_src.transform,
                src_crs=dem_src.crs,
                dst_transform=transform,
                dst_crs=grid.crs,
                resampling=Resampling.bilinear,
            )
    dem_to_use = dem_reproj_path
else:
    print("No reprojection needed.")
    dem_to_use = dem_path


Reprojecting DEM to match grid CRS: EPSG:4547


In [8]:
zs = zonal_stats(
    vectors=grid,
    raster=str(dem_to_use),
    stats=["mean", "std"],
    geojson_out=False
)

grid["elev_mean"] = [z["mean"] for z in zs]
grid["elev_sd"] = [z["std"] for z in zs]


In [9]:
out_path = base_dir / "Jiangsu_grid_500m_ID_with_dem.gpkg"
grid.to_file(out_path, driver="GPKG")

print("Saved:", out_path)


Saved: /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_500m_ID_with_dem.gpkg


In [14]:
import numpy as np
import rasterio
from scipy.ndimage import generic_filter

# 用你刚才得到的 dem_to_use
with rasterio.open(dem_to_use) as src:
    dem = src.read(1).astype("float32")
    profile = src.profile
    nodata = src.nodata

# 把 nodata 变成 NaN 方便处理
if nodata is not None:
    dem[dem == nodata] = np.nan

def tri_func(window):
    """
    window: 3x3 展平成长度为 9 的一维数组
    中间那个元素是中心像元
    """
    center = window[4]
    # 中心是 NaN 的话，直接返回 NaN
    if np.isnan(center):
        return np.nan
    diffs = window - center
    # 邻居是 NaN 的直接忽略（当作 0 贡献）
    diffs[np.isnan(diffs)] = 0.0
    return np.sqrt(np.sum(diffs**2))

# 计算 3x3 TRI
tri = generic_filter(
    dem,
    function=tri_func,
    size=3,
    mode="nearest"   # 边界用最近邻填充
).astype("float32")

# 写出 TRI 栅格
tri_profile = profile.copy()
tri_profile.update(
    dtype="float32",
    nodata=-9999
)

tri[np.isnan(tri)] = -9999

tri_path = dem_src.name.replace(".tif", "_TRI3x3.tif")
with rasterio.open(tri_path, "w", **tri_profile) as dst:
    dst.write(tri, 1)

print("TRI raster saved to:", tri_path)


TRI raster saved to: /Users/wangze/Dropbox/Emi/LandControl/geodata/DEM_JIangsu/江苏省_DEM_30m分辨率_NASA数据_TRI3x3.tif


In [15]:
from rasterstats import zonal_stats

# 还是用你已有的 grid GeoDataFrame
zs_tri = zonal_stats(
    vectors=grid,
    raster=tri_path,
    stats=["mean", "std"],
    geojson_out=False,
    nodata=-9999
)

grid["tri_mean"] = [z["mean"] for z in zs_tri]
grid["tri_sd"] = [z["std"] for z in zs_tri]

# 存一下
out_path = base_dir / "Jiangsu_grid_500m_with_TRI.gpkg"
grid.to_file(out_path, driver="GPKG")
print("Saved grid with TRI to:", out_path)


Saved grid with TRI to: /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_500m_with_TRI.gpkg


In [16]:
import pandas as pd
excel_path = base_dir / "Jiangsu_grid_500m_ID_with_dem.xlsx"

# 去掉 geometry 列，不然 Excel 无法保存 shapely 对象
df = pd.DataFrame(grid.drop(columns="geometry"))

df.to_excel(excel_path, index=False)
print("Saved Excel:", excel_path)

Saved Excel: /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_500m_ID_with_dem.xlsx
